# SPARQL Service Health Check

**Project:** Linked Open Exhibition — NFDI4Culture / Hochschule Hannover (BIM-126-02)  
**AI attribution:** GitHub Copilot (Claude Sonnet 4.6)

**Purpose:** Verify that the SPARQL query service at `https://query.wbworkshop.tibwiki.io/` is reachable and returning results before running upload notebooks.

In [3]:
import os, requests
from dotenv import load_dotenv
from pathlib import Path

load_dotenv(Path('../../.env'))

SPARQL_URL = os.getenv('SPARQL_URL', 'https://query.wbworkshop.tibwiki.io/sparql')
print(f'SPARQL endpoint: {SPARQL_URL}')

SPARQL endpoint: https://query.wbworkshop.tibwiki.io/sparql


## Step 1 — Ping the endpoint

Send a minimal `ASK` query. Expects HTTP 200 and `{ "boolean": true }`.

In [4]:
try:
    resp = requests.get(
        SPARQL_URL,
        params={'query': 'ASK { ?s ?p ?o }', 'format': 'json'},
        headers={'Accept': 'application/sparql-results+json'},
        timeout=15,
    )
    resp.raise_for_status()
    data = resp.json()
    if data.get('boolean') is True:
        print('✓ SPARQL service is UP — ASK query returned true')
    else:
        print(f'⚠ Unexpected response: {data}')
except requests.exceptions.ConnectionError as e:
    print(f'✗ Connection error: {e}')
except requests.exceptions.Timeout:
    print('✗ Request timed out after 15 s')
except Exception as e:
    print(f'✗ Error: {e}')

✓ SPARQL service is UP — ASK query returned true


## Step 2 — Count items in Wikibase

Returns the total number of items (`wd:Q*`) currently in the triplestore.

In [10]:
count_query = '''
SELECT (COUNT(?item) AS ?count)
WHERE {
  ?item ?p ?o .
}
'''
try:
    resp = requests.get(
        SPARQL_URL,
        params={'query': count_query, 'format': 'json'},
        headers={'Accept': 'application/sparql-results+json'},
        timeout=30,
    )
    resp.raise_for_status()
    bindings = resp.json()['results']['bindings']
    count = bindings[0]['count']['value'] if bindings else 'unknown'
    print(f'Items in Wikibase: {count}')
except Exception as e:
    print(f'✗ Error: {e}')


Items in Wikibase: 13955


## Step 3 — Check DNB IDN property exists

Looks up the DNB IDN property ID from `wikibase_property_map.json` and confirms at least one item uses it.

In [12]:
import json, os

MAP_PATH = Path('../wikibase_property_map.json')
wbmap = json.loads(MAP_PATH.read_text(encoding='utf-8'))
P_DNB_IDN = wbmap['properties']['DNB IDN']
print(f'DNB IDN property: {P_DNB_IDN}')

# Construct wdt: prefix from the Wikibase URL (local instance does not auto-resolve it)
WB_URL = os.getenv('WB_URL', 'https://wikibase.wbworkshop.tibwiki.io').rstrip('/')
WDT_PREFIX = f'{WB_URL}/prop/direct/'

sample_query = f'''
PREFIX wdt: <{WDT_PREFIX}>
SELECT ?item ?idn WHERE {{
  ?item wdt:{P_DNB_IDN} ?idn .
}} LIMIT 5
'''
try:
    resp = requests.get(
        SPARQL_URL,
        params={'query': sample_query, 'format': 'json'},
        headers={'Accept': 'application/sparql-results+json'},
        timeout=30,
    )
    resp.raise_for_status()
    bindings = resp.json()['results']['bindings']
    if bindings:
        print(f'✓ Found {len(bindings)} sample item(s) with DNB IDN:')
        for b in bindings:
            qid = b['item']['value'].split('/')[-1]
            idn = b['idn']['value']
            print(f'   {qid} → {idn}')
    else:
        print('⚠ No items found with DNB IDN — data may not be uploaded yet')
except Exception as e:
    print(f'✗ Error: {e}')


DNB IDN property: P7
✓ Found 5 sample item(s) with DNB IDN:
   Q6 → 1375818457
   Q7 → 1375817957
   Q8 → 1347131019
   Q9 → 1357144318
   Q10 → 1361486112
